In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
from matplotlib.colors import Normalize, ListedColormap, BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.patches as mpatches
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import cartopy.io.shapereader as shpreader
from cartopy.feature import ShapelyFeature

# ========= Common Functions =========
def get_extent_from_transform(transform, width, height):
    xmin = transform[2]
    ymax = transform[5]
    xmax = xmin + width * transform[0]
    ymin = ymax + height * transform[4]
    return [xmin, xmax, ymin, ymax]

def reproject_to_wgs84(input_path, resampling=Resampling.nearest):
    with rasterio.open(input_path) as src:
        dst_crs = "EPSG:4326"
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds)
        dst_data = np.zeros((height, width), dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst_data,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=resampling)
    return dst_data, transform, width, height

def base_map(ax):
    states_provinces = cfeature.NaturalEarthFeature('cultural', 'admin_1_states_provinces_lines', '50m', facecolor='none')
    ax.add_feature(cfeature.LAND, alpha=0.1)
    ax.add_feature(cfeature.BORDERS, linestyle='--', lw=0.3, alpha=0.5)
    ax.add_feature(cfeature.LAKES, alpha=0.2)
    ax.add_feature(cfeature.OCEAN, alpha=0.1, zorder=2)
    ax.add_feature(cfeature.COASTLINE, lw=0.2)
    ax.add_feature(cfeature.RIVERS, lw=0.2)
    ax.add_feature(states_provinces, lw=0.2, edgecolor='gray')
    gl = ax.gridlines(draw_labels=True, linestyle=":", linewidth=0.3, color="k")
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = True
    gl.bottom_labels = True
    gl.xlabel_style = {'size': 10, 'rotation': 0, 'ha': 'center', 'va': 'top'}

# ========= Load Data for Panel (a): Forest Area and Edge =========
forest_path = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\Forest_Area_2021_1km.tif"
edge_path = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\Edge_Length_2021_1km.tif"
shpfilename = r"G:\Hangkai\CONUS_Forest_Edge_Mapping\CONUS_shapefile\cb_2018_us_state_5m.shp"
reader = shpreader.Reader(shpfilename)
states = ShapelyFeature(reader.geometries(), ccrs.PlateCarree(), edgecolor='black', facecolor='none', linewidth=0.3)

forest_area, forest_transform, fa_width, fa_height = reproject_to_wgs84(forest_path)
edge_length, _, _, _ = reproject_to_wgs84(edge_path)

forest_area = np.where(forest_area > 0, forest_area, np.nan)
edge_length = np.where(edge_length > 0, edge_length, np.nan)

forest_norm = Normalize(vmin=0, vmax=np.nanpercentile(forest_area, 99))
edge_norm = Normalize(vmin=0, vmax=np.nanpercentile(edge_length, 99))

"""
r = edge_norm(edge_length)
g = forest_norm(forest_area)
b = np.zeros_like(r)
rgb = np.stack([r, g, b], axis=-1)
rgb = np.clip(rgb, 0, 1)
intact_forest_mask = (forest_area > 0) & ((edge_length <= 0) | np.isnan(edge_length))
rgb[intact_forest_mask] = [0, 1, 0]
"""


# Normalize the two variables to [0, 1]
area_scaled = np.clip(forest_norm(forest_area), 0, 1)
edge_scaled = np.clip(edge_norm(edge_length), 0, 1)

# Colorblind-safe bivariate palette
# low area, low edge: light gray
# high area, low edge: blue
# low area, high edge: orange
# high area, high edge: dark purple
color_low  = np.array([0.95, 0.95, 0.95])
color_area = np.array([0.10, 0.45, 0.70])
color_edge = np.array([0.90, 0.45, 0.05])
color_both = np.array([0.35, 0.15, 0.45])

a = area_scaled[..., np.newaxis]
e = edge_scaled[..., np.newaxis]

rgb = (
    (1 - a) * (1 - e) * color_low
    + a * (1 - e) * color_area
    + (1 - a) * e * color_edge
    + a * e * color_both
)

rgb = np.clip(rgb, 0, 1)

# Forest cells without mapped edge are represented by the high-area blue
intact_forest_mask = (
    (forest_area > 0)
    & ((edge_length <= 0) | np.isnan(edge_length))
)
rgb[intact_forest_mask] = color_area

fa_extent = get_extent_from_transform(forest_transform, fa_width, fa_height)

# ========= Load Data for Panel (b): Edge Age =========
edge_age_path = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\LCMAP_edge_age_mean_1km\Edge_Age_mean_2021_990m.tif"
edge_age, transform, width, height = reproject_to_wgs84(edge_age_path, resampling=Resampling.bilinear)
edge_age[edge_age == 0] = np.nan
bounds = rasterio.transform.array_bounds(height, width, transform)
age_extent = [bounds[0], bounds[2], bounds[1], bounds[3]]

vmin = 0
vmax = np.nanpercentile(edge_age, 100)
num_ticks = 6
ticks = np.linspace(vmin, vmax, num_ticks)
tick_labels = [f"{int(t):d}" for t in ticks[:-1]] + [f"{int(vmax):d}+"]

# ========= Create Figure =========
map_extent = [-125, -65, 24, 50]
projection = ccrs.AlbersEqualArea(central_longitude=-96, central_latitude=23, standard_parallels=(29.5, 45.5))

fig, axes = plt.subplots(1, 2, figsize=(12, 6), subplot_kw={'projection': projection}, dpi=1000)

# ========= Panel (a): Forest Area and Edge =========
ax1 = axes[0]
ax1.set_extent(map_extent, crs=ccrs.PlateCarree())
base_map(ax1)
ax1.add_feature(states, linewidth=0.3, edgecolor='black')
nan_mask = np.isnan(forest_area)
nan_mask_3d = np.repeat(nan_mask[:, :, np.newaxis], 3, axis=2)
masked_rgb = np.ma.masked_array(rgb, mask=nan_mask_3d)
ax1.imshow(masked_rgb, extent=fa_extent, origin='upper', transform=ccrs.PlateCarree())
ax1.set_title("(a) Forest Area and Edge Distribution (2021)", fontweight='bold',fontsize=11, loc='left')

# Inset color panel
n = 256
area_vals = np.linspace(0, 1, n)
edge_vals = np.linspace(0, 1, n)

area_grid, edge_grid = np.meshgrid(area_vals, edge_vals)

a_grid = area_grid[..., np.newaxis]
e_grid = edge_grid[..., np.newaxis]

color_grid = (
    (1 - a_grid) * (1 - e_grid) * color_low
    + a_grid * (1 - e_grid) * color_area
    + (1 - a_grid) * e_grid * color_edge
    + a_grid * e_grid * color_both
)

color_grid = np.clip(color_grid, 0, 1)

x0, y0, w, h = 0.8, 0.17, 0.2, 0.2
axins = inset_axes(ax1, width="100%", height="100%", loc='lower right',
    bbox_to_anchor=(x0, y0, w, h), bbox_transform=ax1.transAxes, borderpad=0)
axins.imshow(color_grid, origin='lower', extent=[0, 1, 0, 1], aspect='equal')
axins.set_xlabel('Area (km$^2$)', fontsize=8)
axins.set_ylabel('Edge\nLength\n(km)', fontsize=8)
axins.set_xticks([0, 1])
axins.set_yticks([0, 1])
axins.set_yticklabels(['0', '2.7'], fontsize=8, rotation=90, va='center', ha='right')
axins.set_xticklabels(['0', '1.0'], fontsize=8)
axins.set_title('Forest\nLandscapes', fontsize=8)
axins.tick_params(axis='both', which='major', labelsize=8)
for spine in axins.spines.values():
    spine.set_visible(False)

# ========= Panel (b): Edge Age =========
ax2 = axes[1]
ax2.set_extent(map_extent, crs=ccrs.PlateCarree())
base_map(ax2)
ax2.add_feature(states, linewidth=0.3, edgecolor='black')
im = ax2.imshow(edge_age, origin='upper', extent=age_extent, cmap='cividis',
                vmin=vmin, vmax=vmax, transform=ccrs.PlateCarree())
ax2.set_title(f"(b) Mean Years Since Edge Created (2021)", fontweight='bold',fontsize=11, loc='left')

# Colorbar for panel (b)
cax = inset_axes(ax2, width="4%", height="50%", loc='lower center',
                 bbox_to_anchor=(0.32, 0.1, 1, 1),
                 bbox_transform=ax2.transAxes,
                 borderpad=0)
cbar = plt.colorbar(im, cax=cax, orientation='vertical', ticks=ticks)
cbar.ax.set_yticklabels(tick_labels)
cbar.set_label("Mean Years\nSince Edge Created", fontsize=9)

plt.tight_layout()
plt.savefig(r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\key_outputs\Figure_1.png", dpi=1000, bbox_inches='tight')
plt.show()